# Data Loading

In [ ]:
import pandas as pd
import re
#import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [ ]:
games_df = pd.read_csv('games.csv')
reviews_df = pd.read_csv('dataset.csv')
#reviews_df.head()

games_df.head()
#reviews_df.head()

In [ ]:
listed_genres = set()
for genres in games_df['Genres'].unique().tolist():
    for genre in str(genres).split(','):
        listed_genres.add(genre)


listed_tags = set()
for tags in games_df['Tags'].unique().tolist():
    for tag in str(tags).split(','):
        listed_tags.add(tag)

print(listed_genres)
print(len(listed_genres))

print(listed_tags)
print(len(listed_tags))

In [ ]:
narrative_games = games_df[games_df['Tags'].str.contains('Story Rich') | games_df['Tags'].str.contains('Narrative') | games_df['Tags'].str.contains('Choices Matter')]

narrative_games.head()

In [ ]:
narrative_game_names = narrative_games["AppID"].unique().tolist()

filtered_reviews = reviews_df[reviews_df["app_name"].isin(narrative_game_names)]

print(len(filtered_reviews))
print(len(filtered_reviews["app_name"].unique().tolist()))

filtered_reviews.head()


In [ ]:
# Cleaning up narrative games by columns that are relevant to us
narrative_games_cleaned = narrative_games[['AppID', 'Name', 'Estimated owners', 'Price', 'About the game', 'Positive', 'Negative', 'Score rank', 'Recommendations', 'Categories', 'Genres', 'Tags']]

available_names = filtered_reviews["app_name"].unique()
narrative_games_available = narrative_games_cleaned[narrative_games_cleaned["AppID"].isin(available_names)]

review_count = []
pos_ratio = []

for i in range (len(narrative_games_available)):
    game = narrative_games_available.iloc[i]
    game_reviews = filtered_reviews[filtered_reviews['app_name']==game.get('AppID')]
    game_review_count = len(game_reviews)
    game_postiive_reviews = sum(1 for review in game_reviews['review_score'] if review >0)
    game_pos_ratio = game_postiive_reviews/game_review_count
    review_count.append(game_review_count)
    pos_ratio.append(game_pos_ratio)


narrative_games_available['review_count'] = review_count
narrative_games_available['pos_ratio'] = pos_ratio

narrative_games_available.head()

## Data Cleaning

In [ ]:
#filters for ascii characters
filter_ascii = filtered_reviews["review_text"].replace(to_replace=r'[^\x00-\x7f]', value='', regex=True)

#removes new lines
remove_newline = filter_ascii.replace(to_replace=r'\d', value='', regex=True)

#removes certain characters used in reviews
remove_otherchars = remove_newline.replace(to_replace=r'[,()*/"1234567890+|]', value='', regex=True)

#removes punctuations
replace_punctuations = remove_otherchars.replace(to_replace=r'[.!?]+', value = ' | ', regex=True)

#remove dashes not used to hyphenate words
remove_nonhyphen_dashes = replace_punctuations.replace(to_replace=r'- | -', value=' ', regex=True)

#remove extra whitespace
remove_extra_whitespace = remove_nonhyphen_dashes.replace(to_replace=r' +', value=' ', regex=True)

# assign cleaned entries back to the data frame
new = remove_extra_whitespace
filtered_reviews['review_text'] = new

# apply lowercasing
filtered_reviews['review_text'] = filtered_reviews['review_text'].str.lower()
filtered_reviews

print(filtered_reviews)

In [ ]:
## additional cleaning based on Kaggle User Ali Marjani, PhD
## via https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

print("before removing duplicates: " + str(filtered_reviews.shape))

# drop duplicates
## NOTE: only this actually decreased stuff lmao
filtered_reviews.drop_duplicates(['review_text', 'review_score'], inplace = True)
print("after removing duplicates: " + str(filtered_reviews.shape))


In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

word_tokens = [ (str(review)).split() for review in filtered_reviews['review_text']]

stop_words = set(stopwords.words('english'))
stop_words.add('us')

lemmatized_reviews = []
for tokens in word_tokens:
    lemmatized_reviews.append([lemmatizer.lemmatize(token) for token in tokens if token not in stop_words or token=='not'])

lemmatized_reviews = [" ".join(review) for review in lemmatized_reviews]

filtered_reviews["lemmatized_reviews"] = lemmatized_reviews
filtered_reviews

In [ ]:
# Getting highest rating based on number of reviews and number of positive reviews
top5_df = narrative_games_available.nlargest(6, ["review_count", "pos_ratio"], 'first')
top5_games = []

for i in range (len(top5_df)):
    game = top5_df.iloc[i]
    top5_games.append(game.get("AppID")) #Get all the game names
    

top5_df.head()

In [ ]:
# Did it this way since games with no reviews will register as 0 and will clog up the space for lowest rated games
# We still want a significant amount of reviews to base what is actually bad vs what is underrepresented/underreviewed
cond = narrative_games_available.get("review_count") >= 1000
temp_bot_df = narrative_games_available.where(cond)
bot5_df = temp_bot_df.nsmallest(5, "pos_ratio", 'first')

bot5_games = []

for i in range (len(bot5_df)):
    game = bot5_df.iloc[i]
    bot5_games.append(game.get("AppID")) #Get all the game names

bot5_df.head()
#print(bot5_games)

# Exploratory Data Analysis

In [ ]:
#inserting data from 1 dataset to the next
estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
to_append = [False, False, False]
for i in range (len(filtered_reviews)):
    game = filtered_reviews.iloc[i,1] #sometimes its 1, sometimes its 2,,, idk why 
    if game != last_game:
        estimate = narrative_games.loc[narrative_games["AppID"]==game, "Release date"].iloc[0]
        #for some reason realease date is the header for estimated owners dont ask me
        game_tags = narrative_games.loc[narrative_games["AppID"]==game, "Tags"].iloc[0]
        to_append[0] = "Story Rich" in game_tags
        to_append[1] = "Narrative" in game_tags
        to_append[2] = "Choices Matter" in game_tags
        #same here,, required age = price
        price = narrative_games.loc[narrative_games["AppID"]==game, "Required age"].iloc[0]

    estimated_owners.append(estimate)
    story_rich.append(to_append[0])
    narrative.append(to_append[1])
    choices_matter.append(to_append[2])
    prices.append(price)
    last_game = game    


filtered_reviews["estimated_owners"] = estimated_owners 
filtered_reviews["story_rich"] = story_rich
filtered_reviews["narrative"] = narrative 
filtered_reviews["choices_matter"] = choices_matter 
filtered_reviews["price"] = prices

estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
filtered_reviews.head()

In [ ]:
#Get reviews from top5 games in filtered reviews
all_top5_df = filtered_reviews[filtered_reviews['app_name'].isin(top5_games)]

sampled_top5_array = []

for i in range(len(top5_games)):
    temp_df=filtered_reviews[filtered_reviews["app_name"].str.contains(top5_games[i])]
    sampled_temp_df=temp_df.sample(1000)

    #sample reviews from each
    sampled_top5_array.append(sampled_temp_df)

sampled_top5_df = pd.concat(sampled_top5_array)
sampled_top5_df.head()

In [ ]:
#Get reviews from bottom5 games in filtered reviews
all_bot5_df = filtered_reviews[filtered_reviews['app_name'].isin(bot5_games)]

sampled_bot5_array = []

for i in range(len(bot5_games)):
    temp_df=filtered_reviews[filtered_reviews["app_name"].str.contains(bot5_games[i])]
    if len(temp_df) > 1000:
        sampled_temp_df=temp_df.sample(1000)

        #sample reviews from each
        sampled_bot5_array.append(sampled_temp_df)
    else:
        sampled_bot5_array.append(temp_df)

sampled_bot5_df = pd.concat(sampled_bot5_array)
sampled_bot5_df.head()

In [ ]:
all_topbot_reviews = pd.concat([all_top5_df,all_bot5_df])

sampled_reviews = pd.concat([sampled_top5_df, sampled_bot5_df])

sampled_reviews.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
from wordcloud import WordCloud

In [ ]:
def create_wordcloud(reviews, max_words=500):
    """Create a wordcloud of most common words in a set of tweets"""
    
    # Transform text for WordCloud
    reviews = reviews.dropna()
    reviews = ' '.join(reviews)
    reviews = reviews.replace(' ', ',')
    
    # Generate wordcloud image
    wc = WordCloud(background_color="white", max_words=max_words, colormap='plasma')
    wc.generate(reviews)
    plt.imshow(wc, interpolation='bilinear')
    plt.title('Review Cloud', size=30)
    plt.axis("off")

# Generate a wordcloud
create_wordcloud(filtered_reviews['review_text'])

In [ ]:
create_wordcloud(filtered_reviews['lemmatized_reviews'])

### enable gpu

In [ ]:
import torch

# Check if a CUDA-capable GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This line checks if cuda GPU is available
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# Sentiment Analysis

### Without Sampled Data

In [ ]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(lemmatized_reviews, batch_size=32)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    sentiment = doc._.sentiment
    com_sent.append(sentiment['compound'])
    pos_sent.append(sentiment['pos'])
    neg_sent.append(sentiment['neg'])
    neu_sent.append(sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(filtered_reviews)))
filtered_reviews['com_sent'] = com_sent
filtered_reviews['pos_sent'] = pos_sent
filtered_reviews['neg_sent'] = neg_sent
filtered_reviews['neu_sent'] = neu_sent

filtered_reviews.head()
filtered_reviews.to_csv("output_senti.csv")

### With Sampled Data

In [ ]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(sampled_reviews["lemmatized_reviews"], batch_size=32)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    sentiment = doc._.sentiment
    com_sent.append(sentiment['compound'])
    pos_sent.append(sentiment['pos'])
    neg_sent.append(sentiment['neg'])
    neu_sent.append(sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(sampled_reviews)))

sampled_reviews['com_sent'] = com_sent
sampled_reviews['pos_sent'] = pos_sent
sampled_reviews['neg_sent'] = neg_sent
sampled_reviews['neu_sent'] = neu_sent

sampled_reviews.head()
sampled_reviews.to_csv("output_senti_sampled.csv")

# Topic Modeling

References:

* https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html
* https://maartengr.github.io/BERTopic/getting_started/visualization/visualization.html
* https://maartengr.github.io/BERTopic/getting_started/tips_and_tricks/tips_and_tricks.html

We want to perform topic modeling on:
* reviews from the top 5 games
* reviews from the bottom 5 games
* random sample of reviews across all positive reviews
* random sample of reviews across all negative reviews

NOTE: Not recommended daw to manually stopword removal and lemmatization for BERTopic modeling. There's some tips for removing/reducing stopwords in the third ref above - moi

In [ ]:
TM_SAMPLE_N = 10000

### CRITERIA:
### - sentence form
### - sentence has at least 5 words
### - no need for stopword removal, no need for lematization

def get_review_sentences(reviews_list, min_word_count=5):
    sentences = []
    for i, review in enumerate(reviews_list):
        for sentence in (str(review)).split(" | "):
            if (len(sentence.split()) >= min_word_count):
                sentences.append(("").join(sentence))

    print("Sentences retrieved: " + str(len(sentences)))
    return sentences

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
# from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer


TM_DEFAULT_MIN_TOPIC_SIZE = 200
RANDOM_SEED = 230995 

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

vectorizer_model = CountVectorizer(stop_words="english")

In [ ]:
def perform_topic_modeling(data, min_cluster_size=TM_DEFAULT_MIN_TOPIC_SIZE, data_title=""):
    MIN_TOPIC_COUNT_WANTED = 5
    hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, 
                        metric='euclidean',
                        cluster_selection_method='eom', 
                        prediction_data=True
                    )
    
    topic_model = BERTopic(hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model)
    embeddings = sentence_model.encode(data, show_progress_bar=True, batch_size=32)
    topic_model.fit(data)

    ## IT WILL ERROR IF THERE ARENT ENOUGH TOPICS
    ## so make sure to check
    topic_info = topic_model.get_topic_info()
    topics_count = len(set(topic_info["Topic"])) - 1 # - 1 since there's also a category for outlier topics
    while topics_count < MIN_TOPIC_COUNT_WANTED:
        print("Not enough topics found when min_cluster_size=" + str(hdbscan_model.min_cluster_size) + ". Reducing min cluster size by 5 then rerunning.")
        topic_model.hdbscan_model.set_params(min_cluster_size=(hdbscan_model.min_cluster_size - 5))
        topic_model.fit(data)
        topic_info = topic_model.get_topic_info()
        topics_count = len(set(topic_info["Topic"])) - 1

    data_title = data_title + " | " if (data_title != "") else ""
    vistopics = topic_model.visualize_topics(title=(data_title + "Intertopic Distance Map"))
    visdocus = topic_model.visualize_documents(data, embeddings=embeddings, title=(data_title + "Documents and Topics"))

    return topic_info, vistopics, visdocus


### Positive reviews

In [ ]:
willrecco_df = filtered_reviews[filtered_reviews.review_score==1]
willrecco_reviews = (willrecco_df["review_text"].sample(n=TM_SAMPLE_N, random_state = RANDOM_SEED)).tolist()
willrecco_sentences = get_review_sentences(willrecco_reviews)

In [ ]:
willrecco_topicinfo, willrecco_topicsvis, willrecco_docusvis = perform_topic_modeling(willrecco_sentences, 
                                                                                      data_title=(str(TM_SAMPLE_N) + " Would-Recommend Reviews")
                                                                                      )
willrecco_topicsvis.show()
willrecco_docusvis.show()

### Negative reviews

In [ ]:
notrecco_df = filtered_reviews[filtered_reviews.review_score==-1]
notrecco_reviews = (notrecco_df["review_text"].sample(n=TM_SAMPLE_N, random_state = RANDOM_SEED)).tolist()
notrecco_sentences = get_review_sentences(notrecco_reviews)

In [ ]:
notrecco_topicinfo, notrecco_topicsvis, notrecco_docusvis = perform_topic_modeling(notrecco_sentences, 
                                                                                   data_title=(str(TM_SAMPLE_N) + " Would-Not-Recommend Reviews")
                                                                                   )
notrecco_topicsvis.show()
notrecco_docusvis.show()

### All Positive Reviews from Top 5 Games

In [ ]:
# print(len((all_top5_df[all_top5_df.review_score==1])["review_text"].to_list()))
all_top5_reviews = (all_top5_df[all_top5_df.review_score==1])["review_text"].to_list()
all_top5_sentences = get_review_sentences(all_top5_reviews)

In [ ]:
all_top5_topicinfo, all_top5_topicvis, all_top5_docusvis = perform_topic_modeling(all_top5_sentences, 
                                                                                  min_cluster_size=1000,
                                                                                  data_title="Positive Reviews from Top 5 Games")
all_top5_topicvis.show()
all_top5_docusvis.show()

### All Negative Reviews from Bottom 5 Games

In [ ]:
print(len((all_bot5_df[all_bot5_df.review_score==-1])["review_text"].to_list()))
all_bot5_reviews = (all_bot5_df[all_bot5_df.review_score==-1])["review_text"].to_list()
all_bot5_sentences = get_review_sentences(all_bot5_reviews)

In [ ]:
## this code snippet was initially run with default min cluster size (i.e. min cluster size = 200)
## but topics found were less than 5 until min cluster size was reduced to ___

all_bot5_topicinfo, all_bot5_topicvis, all_bot5_docusvis = perform_topic_modeling(all_bot5_sentences, 
                                                                                  min_cluster_size = 300,
                                                                                  data_title="Negative Reviews from Bottom 5 Games")
all_bot5_topicvis.show()
all_bot5_docusvis.show()

### Topic Modeling Per Game

In [ ]:
## BUG: Portal 2 appears in 'top5_games twice
# temp workaround is this cell

top5_games = list(set(top5_games))
print(top5_games)

In [ ]:
for game in top5_games:
    game_reviews = (all_top5_df[all_top5_df.app_name==game])["review_text"]
    print(game)
    game_sentences = get_review_sentences(game_reviews)



    topic_info, topicvis, docusvis = perform_topic_modeling(game_sentences, 
                                                            min_cluster_size=int(len(game_sentences) * 0.01), 
                                                            data_title=game)

    topicvis.show()
    docusvis.show()

In [ ]:
## BUG: Portal 2 appears in 'top5_games twice
# temp workaround is this cell

bot5_games = list(set(bot5_games))
print(bot5_games)

In [ ]:
for game in bot5_games:
    game_reviews = (all_bot5_df[all_bot5_df.app_name==game])["review_text"]
    print(game)
    # print("---> " + str(len(game_reviews)))
    game_sentences = get_review_sentences(game_reviews)


    
    topic_info, topicvis, docusvis = perform_topic_modeling(game_sentences, 
                                                            min_cluster_size=int(len(game_sentences) * 0.001), 
                                                            data_title=game)

    topicvis.show()
    docusvis.show()